# Module 7 Exercise (Solution): DPO from scratch on a toy verifiable-reward task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/07-rl-for-llms/exercise_solution.ipynb)

Module page: [Module 7: RL for LLMs](https://nsteve2407.github.io/llm-transformers-course/modules/07-rl-for-llms/)

A from-scratch, self-contained pipeline for RLHF-style post-training on a tiny **verifiable-reward** task:
digit sequences, scored by a rule-based reward, no learned reward model required (RLVR-style). Everything
here (the causal transformer, the reward function, PPO's clipped surrogate + KL penalty, and DPO's loss) is
built from raw `torch`/`torch.nn` ops, in the same manual-implementation spirit as `01-dnn-refresher`,
Module 4's from-scratch attention, and Module 6's nanoGPT build.

**Part A**: hyperparameters and documented judgment calls.

**Part B**: the toy setup -- a digit vocabulary and a tiny 2-layer, 2-head causal transformer (the same
causal self-attention pattern as Module 4 / Module 6).

**Part C**: the verifiable reward function -- count of adjacent ascending-or-equal digit pairs -- with
hand-checked unit tests.

**Part D**: reference-policy pretraining (a stand-in for an SFT stage), then freezing a copy as `pi_ref`.

**Part E**: synthetic preference-pair generation -- sample completions from `pi_ref`, score them, and build
(prompt, chosen, rejected) triplets.

**Part F**: a minimal PPO loop -- clipped surrogate objective plus a KL penalty against the frozen `pi_ref`.

**Part G**: the DPO loss, trained directly on the (prompt, chosen, rejected) triplets from the *same* frozen
`pi_ref` checkpoint used for PPO.

**Part H**: evaluation -- `pi_ref` vs. the PPO-tuned policy vs. the DPO-tuned policy on held-out prompts.

**Part I**: an ablation -- weakening PPO's KL coefficient and DPO's beta toward zero, and measuring the
resulting reward hacking / mode collapse.

**Part J**: summary.

In [ ]:
import os
import math
import random
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
random.seed(0)
device = "cpu"  # this task is tiny (13-token vocab, 10-token sequences); CPU is fast enough and keeps
                 # every tensor on the same device as the plain Python reward-function bookkeeping below
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

## Part A: hyperparameters and design choices (documented judgment calls)

This exercise makes several judgment calls the brief explicitly leaves open. Each is documented here so the
reasoning is visible, not just the resulting numbers.

**1. Reward convention: "ascending" means non-decreasing (`b >= a`), not strictly increasing.** A completion
of `COMPLETION_LEN` digits has `COMPLETION_LEN - 1` adjacent pairs; the reward is the count of pairs where
the second digit is greater than *or equal to* the first. This is a deliberate choice: under this convention,
a constant run like `4,4,4,4,4,4` scores the *same* maximum reward as a genuinely increasing run like
`1,2,3,5,7,9`. That is exactly the kind of shallow-verifier loophole real RLVR reward functions can have --
a reward that is a legitimate, verifiable, rule-based proxy for "produce nicely increasing sequences", but
which a policy can satisfy in a cheap, degenerate way. Part I's ablation exploits this loophole directly, so
that "reward hacking" is a real, measured effect rather than an assertion in prose.

**2. No learned Bradley-Terry reward model -- the rule-based reward is used directly for PPO.** The brief
explicitly allows either choice. Training a small scalar-head reward model on the preference pairs would
mostly duplicate work the rule-based `reward_fn` already does exactly and noiselessly, and this task is
squarely in RLVR territory (verifiable rewards replacing a learned RM) -- explicitly in this module's scope.
Skipping the RM keeps the notebook focused on PPO and DPO mechanics, which is what the exercise title
promises. The (prompt, chosen, rejected) preference-pair *construction* in Part E is still needed regardless
-- it feeds DPO -- it just isn't used to train an intermediate RM.

**3. PPO advantage: a running-mean reward baseline, not a value head.** `advantage = reward - baseline`,
where `baseline` is an exponential moving average of the batch-mean reward, broadcast uniformly across every
token position in a completion. This is the simplest of the brief's explicitly-allowed choices. It is a
sequence-level credit assignment (every token in a completion gets the same advantage), which is a common
simplification for toy PPO demonstrations and avoids introducing a second trained network.

**4. Fixed-length completions -- the policy is never asked to sample its own EOS.** Every completion is
exactly `COMPLETION_LEN` digits; `BOS`, the two prompt digits, and the trailing `EOS` are structural, not
sampled. During rollout, sampling logits are masked to the 10 digit tokens only. This sidesteps
variable-length bookkeeping entirely and keeps every sequence exactly `SEQ_LEN` tokens, without changing
what PPO/DPO are actually doing.

**5. The PPO KL penalty is a per-token single-sample log-probability-ratio estimate against the frozen
`pi_ref`** (`E[log pi(a) - log pi_ref(a)]` over the sampled tokens `a`), i.e. literally the "estimated via
log-prob ratio" quantity the brief describes -- not a separately-trained value/critic network's job, and not
the closed-form categorical KL (which would also be tractable here given the tiny vocabulary, but the
log-ratio estimator is what real RLHF/PPO implementations use, since it's the only tractable option once the
vocabulary is large, and reproducing that exact technique is more representative of practice).

**6. Fair comparison between PPO and DPO: identical starting checkpoint and matched gradient-step budget.**
Both the PPO-tuned and DPO-tuned policies are initialized from the exact same frozen `pi_ref` checkpoint (not
independently re-pretrained), and both use `PPO_STEPS` / `DPO_STEPS` gradient steps at the same batch size
(`PPO_EPOCHS=1`, so PPO does exactly one gradient step per rollout batch, exactly like DPO does one gradient
step per triplet batch) -- so any difference in outcome is attributable to the algorithm, not to differing
compute or a differing starting point. This mirrors Module 2's PlainNet/ResNet and Module 6's
baseline/SwiGLU matched-budget comparisons.

**7. Hyperparameters (`kl_coef` for PPO, `beta` for DPO) were tuned empirically, not guessed.** Both
algorithms turned out to have a fairly narrow "well-regularized" operating point at this toy scale: too
little regularization reliably collapses to the reward-hacking loophole in judgment call #1 within a few
hundred gradient steps; too much regularization prevents the policy from moving at all. The chosen baseline
values (`PPO_KL_COEF_BASELINE`, `DPO_BETA_BASELINE`, set below) are the largest regularization strength that
still lets each algorithm learn *something* without collapsing, found by sweeping over multiple seeds.

## Part B: the toy setup -- vocabulary and a tiny causal transformer

The vocabulary is the 10 digits `0`-`9` plus three control tokens: `PAD` (unused here, kept for
convention), `BOS`, and `EOS`. Every sequence has the fixed layout
`[BOS, prompt_0, prompt_1, completion_0, ..., completion_5, EOS]` -- `PROMPT_LEN=2` prompt digits followed by
`COMPLETION_LEN=6` completion digits, for a fixed total length `SEQ_LEN=10`.

The model itself is a tiny 2-layer, 2-head causal Transformer, reusing the exact causal self-attention
pattern from Module 4 / Module 6 (raw `torch`/`torch.nn` ops, no `nn.MultiheadAttention`).

In [ ]:
DIGIT_VOCAB_SIZE = 10
PAD, BOS, EOS = 10, 11, 12
VOCAB_SIZE = 13

PROMPT_LEN = 2
COMPLETION_LEN = 6
SEQ_LEN = 1 + PROMPT_LEN + COMPLETION_LEN + 1  # BOS + prompt + completion + EOS

N_HEAD, N_LAYER = 2, 2  # fixed architecture per the brief, regardless of SMOKE_TEST
BLOCK_SIZE = SEQ_LEN
P_SORTED = 0.4  # fraction of pretraining completions that are partially-sorted rather than fully random

if SMOKE_TEST:
    N_EMBD = 16
    PRETRAIN_STEPS, PRETRAIN_BATCH, PRETRAIN_LR = 150, 16, 3e-3
    N_PAIR_PROMPTS, PAIR_SAMPLES = 40, 4
    PPO_STEPS, PPO_BATCH, PPO_EPOCHS, PPO_LR = 60, 8, 1, 2e-4
    DPO_STEPS, DPO_BATCH, DPO_LR = 60, 8, 3e-4
    EVAL_PROMPTS = 100
    DIVERSITY_PROMPTS, DIVERSITY_K = 10, 8
else:
    N_EMBD = 64
    PRETRAIN_STEPS, PRETRAIN_BATCH, PRETRAIN_LR = 1500, 64, 3e-3
    N_PAIR_PROMPTS, PAIR_SAMPLES = 300, 8
    PPO_STEPS, PPO_BATCH, PPO_EPOCHS, PPO_LR = 600, 32, 1, 2e-4
    DPO_STEPS, DPO_BATCH, DPO_LR = 600, 32, 3e-4
    EVAL_PROMPTS = 200
    DIVERSITY_PROMPTS, DIVERSITY_K = 20, 16

CLIP_EPS = 0.2         # PPO clip range
GRAD_CLIP = 1.0        # gradient-norm clip, standard PPO practice, applied to both PPO and DPO
BASELINE_DECAY = 0.9   # PPO running-mean reward baseline EMA decay

PPO_KL_COEF_BASELINE = 20.0   # regularized PPO run (Part F, Part H) -- large because the per-token
                               # log-ratio KL estimate (judgment call 5) is typically small here (a
                               # restricted 10-way categorical), so it needs a large multiplier to
                               # meaningfully compete with the reward term's gradient magnitude
PPO_KL_COEF_WEAK = 0.0       # weak-regularization ablation (Part I)
DPO_BETA_BASELINE = 5.0      # regularized DPO run (Part G, Part H)
DPO_BETA_WEAK = 0.05         # weak-regularization ablation (Part I)

assert N_EMBD % N_HEAD == 0, "n_embd must be divisible by n_head"
assert PPO_STEPS * PPO_EPOCHS * PPO_BATCH == DPO_STEPS * DPO_BATCH, (
    "PPO and DPO should see a matched number of (gradient step x batch size) examples -- see judgment call 6"
)
print(f"VOCAB_SIZE={VOCAB_SIZE}, SEQ_LEN={SEQ_LEN} (BOS + {PROMPT_LEN} prompt + {COMPLETION_LEN} completion + EOS)")
print(f"N_EMBD={N_EMBD}, N_HEAD={N_HEAD}, N_LAYER={N_LAYER}")
print(f"PRETRAIN_STEPS={PRETRAIN_STEPS}, PPO_STEPS={PPO_STEPS} (x{PPO_EPOCHS} epoch), DPO_STEPS={DPO_STEPS}")
print(f"PPO gradient steps = {PPO_STEPS * PPO_EPOCHS}, DPO gradient steps = {DPO_STEPS} (matched)")

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        assert n_embd % n_head == 0, "n_embd must be divisible by n_head"
        self.n_head = n_head
        self.d_k = n_embd // n_head

        self.W_q = nn.Linear(n_embd, n_embd)
        self.W_k = nn.Linear(n_embd, n_embd)
        self.W_v = nn.Linear(n_embd, n_embd)
        self.W_o = nn.Linear(n_embd, n_embd)

        causal_mask = torch.triu(torch.full((block_size, block_size), float("-inf")), diagonal=1)
        self.register_buffer("causal_mask", causal_mask)  # (block_size, block_size), not a learned Parameter

    def forward(self, x):
        """x: (batch, seq_len, n_embd), seq_len <= block_size. Returns (batch, seq_len, n_embd)."""
        batch_size, seq_len, n_embd = x.shape

        Q = self.W_q(x).view(batch_size, seq_len, self.n_head, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_head, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_head, self.d_k).transpose(1, 2)
        # Q, K, V: (batch, n_head, seq_len, d_k)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (batch, n_head, seq_len, seq_len)
        scores = scores + self.causal_mask[:seq_len, :seq_len]
        attn_weights = F.softmax(scores, dim=-1)
        out = attn_weights @ V  # (batch, n_head, seq_len, d_k)

        out = out.transpose(1, 2).contiguous().view(batch_size, seq_len, n_embd)
        return self.W_o(out)


# Sanity check: causal-mask verification, same technique as Module 4 / Module 6 -- editing the input at
# positions after `modify_pos` must leave every output at position <= modify_pos exactly unchanged.
attn_demo = CausalSelfAttention(N_EMBD, N_HEAD, BLOCK_SIZE)
attn_demo.eval()
modify_pos = BLOCK_SIZE // 2
x1 = torch.randn(1, BLOCK_SIZE, N_EMBD)
x2 = x1.clone()
x2[:, modify_pos + 1:, :] = torch.randn_like(x2[:, modify_pos + 1:, :])
with torch.no_grad():
    out1 = attn_demo(x1)
    out2 = attn_demo(x2)
past_diff = (out1[:, :modify_pos + 1] - out2[:, :modify_pos + 1]).abs().max().item()
future_diff = (out1[:, modify_pos + 1:] - out2[:, modify_pos + 1:]).abs().max().item()
assert past_diff == 0.0, f"causal mask is leaking: past positions changed (max diff {past_diff})"
assert future_diff > 1e-4, "test is vacuous: future positions should differ but don't"
print(f"causal mask verification: past positions unaffected (diff={past_diff}), "
      f"future positions do change (diff={future_diff:.3e}). OK")

In [ ]:
class MLP(nn.Module):
    """Position-wise MLP: Linear -> GELU -> Linear."""

    def __init__(self, n_embd):
        super().__init__()
        self.fc1 = nn.Linear(n_embd, 4 * n_embd)
        self.fc2 = nn.Linear(4 * n_embd, n_embd)

    def forward(self, x):
        return self.fc2(F.gelu(self.fc1(x)))


class Block(nn.Module):
    """Pre-norm decoder block: x = x + attn(LN1(x)); x = x + mlp(LN2(x))."""

    def __init__(self, n_embd, n_head, block_size):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, block_size)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyDigitTransformer(nn.Module):
    """A tiny decoder-only Transformer over the 13-token digit + control vocabulary."""

    def __init__(self, vocab_size, n_embd, n_head, n_layer, block_size):
        super().__init__()
        self.block_size = block_size
        self.tok_embedding = nn.Embedding(vocab_size, n_embd)
        self.pos_embedding = nn.Embedding(block_size, n_embd)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.tok_embedding.weight  # weight tying

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        """idx: (batch, seq_len) token ids, seq_len <= block_size.
        targets: optional (batch, seq_len) next-token ids, elementwise-aligned with idx (caller shifts).
        Returns (logits, loss_or_None).
        """
        batch_size, seq_len = idx.shape
        assert seq_len <= self.block_size, f"sequence length {seq_len} exceeds block_size {self.block_size}"

        positions = torch.arange(seq_len, device=idx.device)
        x = self.tok_embedding(idx) + self.pos_embedding(positions)[None, :, :]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.head(x)  # (batch, seq_len, vocab_size)

        loss = None
        if targets is not None:
            # .reshape (not .view): weight tying makes the head's output non-contiguous.
            loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), targets.reshape(-1))
        return logits, loss


# Sanity check: output shape and parameter count.
torch.manual_seed(0)
model_demo = TinyDigitTransformer(VOCAB_SIZE, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE)
idx_demo = torch.randint(0, VOCAB_SIZE, (4, SEQ_LEN))
logits_demo, _ = model_demo(idx_demo)
assert logits_demo.shape == (4, SEQ_LEN, VOCAB_SIZE)
n_params = sum(p.numel() for p in model_demo.parameters())
print(f"TinyDigitTransformer output shape OK: {logits_demo.shape}; {n_params:,} parameters")

## Part C: the verifiable rule-based reward function

`reward_fn` takes a completion (a list of `COMPLETION_LEN` digits) and returns the count of adjacent pairs
`(a, b)` with `b >= a` -- see judgment call #1 in Part A for why the convention is non-strict ("ascending or
equal") rather than strictly increasing.

In [ ]:
def reward_fn(completion_digits):
    """completion_digits: list[int] (or any indexable sequence of ints), the digits of one completion.
    Returns the count of adjacent pairs (a, b) with b >= a (non-decreasing). Pure function, no side effects.
    """
    if len(completion_digits) < 2:
        return 0
    return sum(1 for a, b in zip(completion_digits, completion_digits[1:]) if b >= a)

In [ ]:
# Hand-checked unit tests.
assert reward_fn([1, 3, 5, 7]) == 3          # strictly increasing: all 3 adjacent pairs count
assert reward_fn([9, 5, 3, 1]) == 0          # strictly decreasing: no adjacent pair counts
assert reward_fn([4, 4, 4, 4]) == 3          # constant run: every pair is "ascending or equal" -- see
                                              # judgment call #1, this is the exploitable degenerate case
assert reward_fn([1, 1, 2, 2]) == 3          # mixed equal/increasing: all 3 pairs count
assert reward_fn([5, 3, 3, 7]) == 2          # (5,3) no, (3,3) yes, (3,7) yes
assert reward_fn([0, 9, 0, 9]) == 2          # (0,9) yes, (9,0) no, (0,9) yes
assert reward_fn([7]) == 0                   # single element: no adjacent pairs
assert reward_fn([]) == 0                    # empty: no adjacent pairs
assert reward_fn(list(range(10))) == 9       # fully ascending 10-digit run: all 9 pairs count (the max
                                              # achievable reward for a 10-digit sequence)
assert reward_fn([3] * 10) == 9              # a fully-repeated 10-digit run scores the SAME maximum --
                                              # exactly the reward-hacking loophole judgment call #1 predicts
print("reward_fn unit tests: all 10 assertions passed")

## Part D: reference-policy pretraining (a stand-in for SFT), then freezing `pi_ref`

The tiny transformer is pretrained via next-token cross-entropy on a synthetic mix of random digit
sequences and partially-sorted digit sequences (a stand-in for an SFT stage: it gives the model *some*
notion that ascending runs are "normal", without making the reference policy near-optimal on the reward --
that's what PPO and DPO are for). Once pretrained, a frozen copy is saved as `pi_ref`.

In [ ]:
def sample_prompt_batch(batch_size):
    return torch.randint(0, DIGIT_VOCAB_SIZE, (batch_size, PROMPT_LEN))


def sample_pretrain_sequence():
    """One (prompt, completion) pair for the synthetic pretraining corpus."""
    prompt = [random.randint(0, 9) for _ in range(PROMPT_LEN)]
    if random.random() < P_SORTED:
        cur = random.randint(0, 9)
        completion = [cur]
        for _ in range(COMPLETION_LEN - 1):
            cur = min(9, cur + random.choice([0, 1, 1, 2]))
            completion.append(cur)
    else:
        completion = [random.randint(0, 9) for _ in range(COMPLETION_LEN)]
    return prompt, completion


def get_pretrain_batch(batch_size):
    """Returns a (batch_size, SEQ_LEN) tensor of full [BOS, prompt, completion, EOS] sequences."""
    prompts, completions = [], []
    for _ in range(batch_size):
        p, c = sample_pretrain_sequence()
        prompts.append(p)
        completions.append(c)
    prompts = torch.tensor(prompts, dtype=torch.long)
    completions = torch.tensor(completions, dtype=torch.long)
    bos = torch.full((batch_size, 1), BOS, dtype=torch.long)
    eos = torch.full((batch_size, 1), EOS, dtype=torch.long)
    return torch.cat([bos, prompts, completions, eos], dim=1)


# Quick illustration of the two kinds of synthetic pretraining data.
random.seed(0)
example_sorted = None
example_random = None
for _ in range(200):
    p, c = sample_pretrain_sequence()
    if c == sorted(c) and example_sorted is None:
        example_sorted = (p, c)
    if example_random is None and reward_fn(c) <= 1:
        example_random = (p, c)
random.seed(0)  # restore, so the corpus below is reproducible regardless of how many examples we peeked at
print("example partially-sorted completion:", example_sorted)
print("example low-reward (mostly random) completion:", example_random)

In [ ]:
def pretrain(model, steps, batch_size, lr):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    losses = []
    for step in range(steps):
        seq = get_pretrain_batch(batch_size)
        inputs, targets = seq[:, :-1], seq[:, 1:]
        _, loss = model(inputs, targets)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


torch.manual_seed(0)
pi_model = TinyDigitTransformer(VOCAB_SIZE, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE)
pretrain_losses = pretrain(pi_model, PRETRAIN_STEPS, PRETRAIN_BATCH, PRETRAIN_LR)

plt.figure()
plt.plot(pretrain_losses)
plt.xlabel("pretraining step")
plt.ylabel("next-token cross-entropy loss")
plt.title("Reference-policy pretraining loss")
plt.show()

assert pretrain_losses[-1] < pretrain_losses[0], "pretraining loss did not decrease"
print(f"pretrain loss: {pretrain_losses[0]:.3f} -> {pretrain_losses[-1]:.3f}")

In [ ]:
# Freeze a copy of the pretrained model as pi_ref. Every parameter has requires_grad=False, so no
# optimizer step anywhere in this notebook can ever move it -- this is the reference policy that PPO's
# KL penalty and DPO's loss are both computed against.
pi_ref = copy.deepcopy(pi_model)
for p in pi_ref.parameters():
    p.requires_grad_(False)
pi_ref.eval()
assert all(not p.requires_grad for p in pi_ref.parameters()), "pi_ref must be fully frozen"

# Save the pretrained weights so PPO and DPO can each start from the exact same checkpoint (judgment call 6).
pretrained_state = copy.deepcopy(pi_model.state_dict())
# Snapshot pi_ref's own weights too, so we can verify at the very end of the notebook that nothing --
# not PPO, not DPO, not the ablation runs -- ever mutated pi_ref.
pi_ref_snapshot = {k: v.clone() for k, v in pi_ref.state_dict().items()}

print("pi_ref frozen: requires_grad is False for all", sum(1 for _ in pi_ref.parameters()), "parameter tensors")

In [ ]:
def make_policy_from_pretrained():
    """A fresh, trainable TinyDigitTransformer initialized from the pretrained (pre-RL) checkpoint."""
    policy = TinyDigitTransformer(VOCAB_SIZE, N_EMBD, N_HEAD, N_LAYER, BLOCK_SIZE)
    policy.load_state_dict(pretrained_state)
    return policy


def sample_completions(model, prompts, temperature=1.0):
    """Autoregressively sample COMPLETION_LEN digit tokens (masking control tokens) given prompts.
    prompts: (batch, PROMPT_LEN) long tensor. Returns (completions, logprobs), each (batch, COMPLETION_LEN).
    """
    batch_size = prompts.shape[0]
    bos = torch.full((batch_size, 1), BOS, dtype=torch.long)
    seq = torch.cat([bos, prompts], dim=1)
    logprob_steps = []
    for _ in range(COMPLETION_LEN):
        logits, _ = model(seq)
        next_logits = logits[:, -1, :].clone() / temperature
        next_logits[:, DIGIT_VOCAB_SIZE:] = float("-inf")  # only ever sample digit tokens
        next_logp = F.log_softmax(next_logits, dim=-1)
        next_token = torch.multinomial(next_logp.exp(), num_samples=1)  # (batch, 1)
        logprob_steps.append(torch.gather(next_logp, 1, next_token))
        seq = torch.cat([seq, next_token], dim=1)
    completions = seq[:, PROMPT_LEN + 1:]
    logprobs = torch.cat(logprob_steps, dim=1)
    return completions, logprobs


def sequence_logprobs(model, prompts, completions):
    """Per-token teacher-forced log pi(completion_i | prompt, completion_<i>) under `model`.
    prompts: (batch, PROMPT_LEN), completions: (batch, COMPLETION_LEN). Returns (batch, COMPLETION_LEN).
    """
    batch_size = prompts.shape[0]
    bos = torch.full((batch_size, 1), BOS, dtype=torch.long)
    eos = torch.full((batch_size, 1), EOS, dtype=torch.long)
    seq = torch.cat([bos, prompts, completions, eos], dim=1)  # (batch, SEQ_LEN)
    logits, _ = model(seq)
    # logits[:, PROMPT_LEN + i, :] predicts completion token i (see Part B's sequence layout).
    comp_logits = logits[:, PROMPT_LEN:PROMPT_LEN + COMPLETION_LEN, :].clone()
    comp_logits[:, :, DIGIT_VOCAB_SIZE:] = float("-inf")  # match sample_completions's digit-only masking,
                                                           # so this is the log-prob under the SAME
                                                           # restricted-support distribution actually used
                                                           # for sampling -- required for PPO's ratio and
                                                           # DPO's log-prob sums to be self-consistent
    comp_logp = F.log_softmax(comp_logits, dim=-1)
    return torch.gather(comp_logp, 2, completions.unsqueeze(-1)).squeeze(-1)


# Sanity check: a completion's own per-token logprobs from sample_completions (computed incrementally
# during sampling) should match sequence_logprobs's teacher-forced recomputation of the same completion.
torch.manual_seed(1)
_prompts_check = sample_prompt_batch(8)
with torch.no_grad():
    _completions_check, _lp_sampled = sample_completions(pi_ref, _prompts_check)
    _lp_recomputed = sequence_logprobs(pi_ref, _prompts_check, _completions_check)
_max_diff = (_lp_sampled - _lp_recomputed).abs().max().item()
assert _max_diff < 1e-4, f"sample_completions and sequence_logprobs disagree (max diff {_max_diff})"
print(f"sample_completions vs. sequence_logprobs agreement: max diff {_max_diff:.2e}. OK")

## Part E: synthetic preference-pair generation

For `N_PAIR_PROMPTS` random prompts, sample `PAIR_SAMPLES` completions each from `pi_ref`, score every
completion with `reward_fn`, and keep the highest-reward completion as "chosen" and the lowest-reward as
"rejected". Prompts where the best and worst sampled completions tie are dropped -- a tied pair carries no
preference signal and would make the DPO loss's target direction meaningless.

In [ ]:
def build_preference_pairs(model, n_prompts, k_samples):
    pairs = []
    prompts = sample_prompt_batch(n_prompts)
    with torch.no_grad():
        for i in range(n_prompts):
            prompt_i = prompts[i:i + 1].expand(k_samples, -1)
            completions, _ = sample_completions(model, prompt_i, temperature=1.0)
            rewards = [reward_fn(c.tolist()) for c in completions]
            best = max(range(k_samples), key=lambda j: rewards[j])
            worst = min(range(k_samples), key=lambda j: rewards[j])
            if rewards[best] > rewards[worst]:  # drop ties -- no preference signal
                pairs.append((prompts[i].tolist(), completions[best].tolist(), completions[worst].tolist(),
                              rewards[best], rewards[worst]))
    return pairs


torch.manual_seed(2)
preference_pairs = build_preference_pairs(pi_ref, N_PAIR_PROMPTS, PAIR_SAMPLES)
reward_gaps = [chosen_r - rejected_r for _, _, _, chosen_r, rejected_r in preference_pairs]

assert len(preference_pairs) > 0, "no valid (non-tied) preference pairs were generated"
print(f"generated {len(preference_pairs)} valid preference pairs out of {N_PAIR_PROMPTS} prompts "
      f"({PAIR_SAMPLES} samples each)")
print(f"reward gap (chosen - rejected): mean={sum(reward_gaps)/len(reward_gaps):.2f}, "
      f"min={min(reward_gaps)}, max={max(reward_gaps)}")
print("example pair:", {"prompt": preference_pairs[0][0], "chosen": preference_pairs[0][1],
                         "chosen_reward": preference_pairs[0][3], "rejected": preference_pairs[0][2],
                         "rejected_reward": preference_pairs[0][4]})

In [ ]:
def sample_preference_batch(pairs, batch_size):
    batch = random.choices(pairs, k=batch_size)
    prompts = torch.tensor([p for p, _, _, _, _ in batch], dtype=torch.long)
    chosen = torch.tensor([c for _, c, _, _, _ in batch], dtype=torch.long)
    rejected = torch.tensor([r for _, _, r, _, _ in batch], dtype=torch.long)
    return prompts, chosen, rejected

## Part F: a minimal PPO loop (clipped surrogate objective + KL penalty vs. `pi_ref`)

Each PPO step: sample a batch of completions from the *current* policy (this is `old_logprobs`, no
gradient), score them with `reward_fn`, form a per-token advantage as `reward - running_mean_baseline`
(judgment call #3), then take one gradient step on the clipped surrogate objective plus a KL penalty against
the frozen `pi_ref` (judgment call #5). `compute_ppo_loss` is the function that does the actual PPO math --
implement it below.

In [ ]:
def compute_ppo_loss(new_logprobs, old_logprobs, ref_logprobs, advantages, clip_eps, kl_coef):
    """All arguments except clip_eps/kl_coef are (batch, COMPLETION_LEN) tensors.
    new_logprobs: log pi_theta(a_t | ...), WITH gradient (current, being-optimized policy).
    old_logprobs: log pi_theta_old(a_t | ...), no gradient (policy at rollout time, before this update).
    ref_logprobs: log pi_ref(a_t | ...), no gradient (frozen reference policy).
    advantages: per-token advantage estimates (here: reward - running-mean baseline, broadcast over tokens).
    Returns (loss, info) where loss is a scalar tensor to minimize and info is a dict of floats for logging.
    """
    ratio = torch.exp(new_logprobs - old_logprobs)
    surr1 = ratio * advantages
    surr2 = torch.clamp(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * advantages
    clipped_surrogate = torch.min(surr1, surr2).mean()

    kl_estimate = (new_logprobs - ref_logprobs).mean()  # per-token log-ratio KL estimate (judgment call 5)

    loss = -clipped_surrogate + kl_coef * kl_estimate
    clip_frac = ((ratio < 1 - clip_eps) | (ratio > 1 + clip_eps)).float().mean().item()
    info = {"clipped_surrogate": clipped_surrogate.item(), "kl": kl_estimate.item(), "clip_frac": clip_frac}
    return loss, info

In [ ]:
# Unit tests for compute_ppo_loss, exercising the SAME function the PPO training loop below calls.
# old_logprobs = 0 for every entry, so ratio = exp(new_logprobs) = ratio_test directly.
ratio_test = torch.tensor([0.5, 1.0, 1.5, 3.0])
old_lp_test = torch.zeros(4)
new_lp_test = torch.log(ratio_test)

# 1) Positive advantage: clipping should cap the *upside* only (ratio > 1+eps gets clamped down), matching
#    the PPO paper's min(unclipped, clipped) formula -- not naive clamping of the ratio itself.
loss_pos, _ = compute_ppo_loss(new_lp_test, old_lp_test, ref_logprobs=new_lp_test,
                                advantages=torch.ones(4), clip_eps=0.2, kl_coef=0.0)
expected_obj_pos = torch.tensor([0.5, 1.0, 1.2, 1.2])  # min(ratio*A, clip(ratio)*A), A=+1
assert abs(loss_pos.item() - (-expected_obj_pos.mean().item())) < 1e-5, \
    f"positive-advantage clipped surrogate mismatch: got loss {loss_pos.item()}"

# 2) Negative advantage: PPO's clip is asymmetric by design -- it should cap the ratio's move toward 1
#    (which would make the objective look artificially *better*), but NOT cap it from moving further away
#    (which already looks worse, so there's no need to clip). This is the subtlety that distinguishes a
#    genuine clipped surrogate from a naive torch.clamp(ratio, ...) * advantage.
loss_neg, _ = compute_ppo_loss(new_lp_test, old_lp_test, ref_logprobs=new_lp_test,
                                advantages=-torch.ones(4), clip_eps=0.2, kl_coef=0.0)
expected_obj_neg = torch.tensor([-0.8, -1.0, -1.5, -3.0])  # min(ratio*A, clip(ratio)*A), A=-1
assert abs(loss_neg.item() - (-expected_obj_neg.mean().item())) < 1e-5, \
    f"negative-advantage clipped surrogate mismatch: got loss {loss_neg.item()}"

# 3) The KL term is really being added, with the correct sign: with advantages=0 (so the surrogate
#    contributes nothing), increasing kl_coef from 0 to 1 should change the loss by exactly the mean KL
#    estimate (here mean(log(ratio_test)) computed against a ref_logprobs of 0).
loss_kl0, _ = compute_ppo_loss(new_lp_test, old_lp_test, ref_logprobs=torch.zeros(4),
                                advantages=torch.zeros(4), clip_eps=0.2, kl_coef=0.0)
loss_kl1, _ = compute_ppo_loss(new_lp_test, old_lp_test, ref_logprobs=torch.zeros(4),
                                advantages=torch.zeros(4), clip_eps=0.2, kl_coef=1.0)
expected_kl = new_lp_test.mean().item()
assert abs(loss_kl0.item()) < 1e-6, "with advantages=0, kl_coef=0 loss should be exactly 0"
assert abs(loss_kl1.item() - expected_kl) < 1e-5, \
    f"KL term not contributing correctly: expected {expected_kl}, got {loss_kl1.item()}"

print("compute_ppo_loss unit tests: positive-advantage clip, negative-advantage clip, and KL contribution "
      "all verified")

In [ ]:
def run_ppo(policy, pi_ref_model, steps, batch_size, epochs, kl_coef, lr, baseline_init,
            clip_eps=CLIP_EPS, baseline_decay=BASELINE_DECAY, grad_clip=GRAD_CLIP):
    optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
    baseline = baseline_init
    reward_history = []
    for step in range(steps):
        prompts = sample_prompt_batch(batch_size)
        with torch.no_grad():
            completions, old_logprobs = sample_completions(policy, prompts, temperature=1.0)
            ref_logprobs = sequence_logprobs(pi_ref_model, prompts, completions)
            rewards = torch.tensor([float(reward_fn(c.tolist())) for c in completions])
            baseline = baseline_decay * baseline + (1 - baseline_decay) * rewards.mean().item()
            advantages = (rewards - baseline).unsqueeze(1).expand(-1, COMPLETION_LEN)
        for _ in range(epochs):
            new_logprobs = sequence_logprobs(policy, prompts, completions)
            loss, info = compute_ppo_loss(new_logprobs, old_logprobs, ref_logprobs, advantages,
                                           clip_eps, kl_coef)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip)
            optimizer.step()
        reward_history.append(rewards.mean().item())
    return reward_history


torch.manual_seed(3)
ppo_policy = make_policy_from_pretrained()
ref_reward_estimate = sum(reward_fn(c) for _, c in (sample_pretrain_sequence() for _ in range(200))) / 200
ppo_reward_history = run_ppo(ppo_policy, pi_ref, PPO_STEPS, PPO_BATCH, PPO_EPOCHS,
                              kl_coef=PPO_KL_COEF_BASELINE, lr=PPO_LR, baseline_init=ref_reward_estimate)

plt.figure()
plt.plot(ppo_reward_history)
plt.xlabel("PPO step")
plt.ylabel("mean rollout reward")
plt.title(f"PPO training reward per step (kl_coef={PPO_KL_COEF_BASELINE})")
plt.show()

print(f"PPO rollout reward: first={ppo_reward_history[0]:.3f}, "
      f"last 10-step mean={sum(ppo_reward_history[-10:]) / min(10, len(ppo_reward_history)):.3f}")

In [ ]:
# pi_ref must still be untouched after training the PPO policy: every parameter tensor bit-for-bit equal
# to the snapshot taken right after freezing it in Part D.
for k, v in pi_ref.state_dict().items():
    assert torch.equal(v, pi_ref_snapshot[k]), f"pi_ref parameter '{k}' changed during PPO training!"
print("pi_ref frozen-ness verified after PPO training: every parameter tensor unchanged")

## Part G: the DPO loss, on the same frozen `pi_ref`

DPO trains directly on the (prompt, chosen, rejected) triplets from Part E -- no rollouts, no reward
function calls during training, just a classification-style loss comparing the trainable policy's and the
frozen `pi_ref`'s relative preference for chosen vs. rejected:

`-log sigmoid(beta * ((logpi(chosen) - logpi_ref(chosen)) - (logpi(rejected) - logpi_ref(rejected))))`

`dpo_policy` starts from the exact same pretrained checkpoint as `ppo_policy` above.

In [ ]:
def dpo_loss(chosen_logp, rejected_logp, chosen_ref_logp, rejected_ref_logp, beta):
    """All four arguments are (batch,) scalar SEQUENCE log-probabilities (already summed over tokens).
    chosen_logp, rejected_logp: under the trainable policy, WITH gradient.
    chosen_ref_logp, rejected_ref_logp: under the frozen pi_ref, no gradient.
    Returns the scalar DPO loss (mean over the batch).
    """
    policy_logratio = chosen_logp - rejected_logp
    ref_logratio = chosen_ref_logp - rejected_ref_logp
    logits = beta * (policy_logratio - ref_logratio)
    return -F.logsigmoid(logits).mean()

In [ ]:
# Unit tests for dpo_loss, exercising the SAME function DPO training calls.
# 1) If the policy prefers "chosen" much more strongly than pi_ref does, the margin is large and positive,
#    so the loss (which drives that margin to +infinity) should already be small.
loss_good = dpo_loss(torch.tensor([0.0]), torch.tensor([-5.0]), torch.tensor([-1.0]), torch.tensor([-1.0]),
                      beta=1.0)
assert loss_good.item() < 0.1, f"expected near-zero loss when policy strongly prefers chosen, got {loss_good.item()}"

# 2) The reverse case (policy prefers the REJECTED completion) should give a much larger loss.
loss_bad = dpo_loss(torch.tensor([-5.0]), torch.tensor([0.0]), torch.tensor([-1.0]), torch.tensor([-1.0]),
                     beta=1.0)
assert loss_bad.item() > loss_good.item() + 1.0, \
    f"loss should be much higher when the policy prefers the rejected completion: {loss_bad.item()} vs {loss_good.item()}"

# 3) When policy == ref exactly (no learning has happened yet), the margin is exactly 0 regardless of beta,
#    so the loss should be exactly -log(0.5) = log(2) -- the DPO loss at initialization.
loss_init = dpo_loss(torch.tensor([-2.0]), torch.tensor([-3.0]), torch.tensor([-2.0]), torch.tensor([-3.0]),
                      beta=5.0)
assert abs(loss_init.item() - math.log(2)) < 1e-5, f"expected log(2) at policy==ref, got {loss_init.item()}"

print(f"dpo_loss unit tests: loss_good={loss_good.item():.4f}, loss_bad={loss_bad.item():.4f}, "
      f"loss_at_init={loss_init.item():.4f} (== log(2)={math.log(2):.4f}). All checks passed.")

In [ ]:
def run_dpo(policy, pi_ref_model, pairs, steps, batch_size, beta, lr, grad_clip=GRAD_CLIP):
    optimizer = torch.optim.AdamW(policy.parameters(), lr=lr)
    loss_history = []
    for step in range(steps):
        prompts, chosen, rejected = sample_preference_batch(pairs, batch_size)
        with torch.no_grad():
            chosen_ref_logp = sequence_logprobs(pi_ref_model, prompts, chosen).sum(dim=1)
            rejected_ref_logp = sequence_logprobs(pi_ref_model, prompts, rejected).sum(dim=1)
        chosen_logp = sequence_logprobs(policy, prompts, chosen).sum(dim=1)
        rejected_logp = sequence_logprobs(policy, prompts, rejected).sum(dim=1)
        loss = dpo_loss(chosen_logp, rejected_logp, chosen_ref_logp, rejected_ref_logp, beta)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), grad_clip)
        optimizer.step()
        loss_history.append(loss.item())
    return loss_history


torch.manual_seed(4)
dpo_policy = make_policy_from_pretrained()
dpo_loss_history = run_dpo(dpo_policy, pi_ref, preference_pairs, DPO_STEPS, DPO_BATCH,
                            beta=DPO_BETA_BASELINE, lr=DPO_LR)

plt.figure()
plt.plot(dpo_loss_history)
plt.xlabel("DPO step")
plt.ylabel("DPO loss")
plt.title(f"DPO training loss (beta={DPO_BETA_BASELINE})")
plt.show()

assert dpo_loss_history[-1] < dpo_loss_history[0], "DPO loss did not decrease"
print(f"DPO loss: {dpo_loss_history[0]:.4f} -> {dpo_loss_history[-1]:.4f}")

In [ ]:
# pi_ref must still be untouched after DPO training too.
for k, v in pi_ref.state_dict().items():
    assert torch.equal(v, pi_ref_snapshot[k]), f"pi_ref parameter '{k}' changed during DPO training!"
print("pi_ref frozen-ness verified after DPO training: every parameter tensor unchanged")

## Part H: evaluation -- `pi_ref` vs. PPO-tuned vs. DPO-tuned, on held-out prompts

`eval_mean_reward` samples one completion per held-out prompt (fresh, never-seen-before prompts, sampled
with a different random seed than the preference-pair prompts in Part E) and reports the mean reward.
`eval_diversity` measures, for a small set of prompts, how many *distinct* completions a model produces
across repeated sampling of the same prompt -- this will matter more in Part I's ablation than here.

In [ ]:
def eval_mean_reward(model, n_prompts):
    prompts = sample_prompt_batch(n_prompts)
    with torch.no_grad():
        completions, _ = sample_completions(model, prompts, temperature=1.0)
    rewards = [reward_fn(c.tolist()) for c in completions]
    return sum(rewards) / len(rewards), rewards


def eval_diversity(model, n_prompts, k):
    """For n_prompts distinct prompts, sample k completions each. Returns:
    - mean unique-completion ratio per prompt (1.0 = every sample distinct, near 0 = mode collapse)
    - degenerate rate: fraction of ALL sampled completions that are a single repeated digit
    """
    prompts = sample_prompt_batch(n_prompts)
    unique_ratios = []
    degenerate_count, total_count = 0, 0
    with torch.no_grad():
        for i in range(n_prompts):
            prompt_i = prompts[i:i + 1].expand(k, -1)
            completions, _ = sample_completions(model, prompt_i, temperature=1.0)
            comp_list = [tuple(c.tolist()) for c in completions]
            unique_ratios.append(len(set(comp_list)) / k)
            for c in comp_list:
                total_count += 1
                if len(set(c)) == 1:
                    degenerate_count += 1
    return sum(unique_ratios) / len(unique_ratios), degenerate_count / total_count


torch.manual_seed(5)  # a fresh seed for held-out evaluation prompts -- disjoint in practice from Part E's
ref_mean_reward, ref_rewards = eval_mean_reward(pi_ref, EVAL_PROMPTS)
ppo_mean_reward, ppo_rewards = eval_mean_reward(ppo_policy, EVAL_PROMPTS)
dpo_mean_reward, dpo_rewards = eval_mean_reward(dpo_policy, EVAL_PROMPTS)

print(f"held-out mean reward: pi_ref={ref_mean_reward:.3f}, PPO={ppo_mean_reward:.3f}, DPO={dpo_mean_reward:.3f}")
if not SMOKE_TEST:
    # At full scale both algorithms reliably improve on pi_ref; at SMOKE_TEST scale there simply isn't
    # enough training budget for this to be a reliable comparison (see the same caveat in Part I).
    assert ppo_mean_reward > ref_mean_reward, "PPO should improve mean reward over pi_ref"
    assert dpo_mean_reward > ref_mean_reward, "DPO should improve mean reward over pi_ref"

plt.figure()
plt.hist(ref_rewards, bins=range(0, COMPLETION_LEN + 1), alpha=0.5, label="pi_ref", density=True)
plt.hist(ppo_rewards, bins=range(0, COMPLETION_LEN + 1), alpha=0.5, label="PPO-tuned", density=True)
plt.hist(dpo_rewards, bins=range(0, COMPLETION_LEN + 1), alpha=0.5, label="DPO-tuned", density=True)
plt.xlabel("reward (adjacent ascending-or-equal pairs)")
plt.ylabel("density")
plt.title("Reward distribution on held-out prompts")
plt.legend()
plt.show()

Both tuned policies improve on `pi_ref`'s mean reward while keeping most of its output diversity -- but
getting there took a much larger KL coefficient for PPO (`PPO_KL_COEF_BASELINE`) than beta for DPO
(`DPO_BETA_BASELINE`) relative to each algorithm's own natural loss scale (see judgment call 7 in Part A).
At this toy scale, PPO's on-policy rollouts found the reward-hacking loophole from judgment call 1 easily
and needed strong, carefully-tuned regularization to avoid collapsing into it well before this training
budget ran out; DPO's classification-style loss over static preference pairs was noticeably easier to keep
in a "learns something, doesn't collapse" regime. That gap in how fiddly the regularization is to get right
is itself a real, small-scale illustration of the widely-reported practical experience that on-policy RLHF
(PPO) is harder to tune reliably than DPO's simpler offline objective -- and Part I now pushes both algorithms
past that regularization boundary on purpose.

## Part I: ablation -- weakening KL/beta causes reward hacking and mode collapse

Two more policies are trained from the exact same pretrained checkpoint: `ppo_policy_weak`
(`kl_coef=PPO_KL_COEF_WEAK=0`, i.e. no KL penalty at all) and `dpo_policy_weak`
(`beta=DPO_BETA_WEAK`, a much smaller beta than the baseline). Recall from judgment call #1 that
`reward_fn`'s max score is achieved *equally* by a genuinely ascending run and by a degenerate
all-one-digit run -- so an unregularized policy has no reward-based reason to prefer the diverse solution
over the cheap, repeated-digit one. We measure this directly with `eval_diversity`'s unique-completion ratio
and degenerate rate.

In [ ]:
torch.manual_seed(6)
ppo_policy_weak = make_policy_from_pretrained()
run_ppo(ppo_policy_weak, pi_ref, PPO_STEPS, PPO_BATCH, PPO_EPOCHS,
        kl_coef=PPO_KL_COEF_WEAK, lr=PPO_LR, baseline_init=ref_reward_estimate)

torch.manual_seed(6)
dpo_policy_weak = make_policy_from_pretrained()
run_dpo(dpo_policy_weak, pi_ref, preference_pairs, DPO_STEPS, DPO_BATCH, beta=DPO_BETA_WEAK, lr=DPO_LR)

print("trained ppo_policy_weak (kl_coef=0) and dpo_policy_weak (beta="
      f"{DPO_BETA_WEAK}) from the same pretrained checkpoint")

In [ ]:
torch.manual_seed(7)  # same eval protocol/seed for every model, for a fair before/after comparison
ablation_configs = [
    ("pi_ref", pi_ref),
    (f"PPO baseline (kl_coef={PPO_KL_COEF_BASELINE})", ppo_policy),
    (f"PPO weak (kl_coef={PPO_KL_COEF_WEAK})", ppo_policy_weak),
    (f"DPO baseline (beta={DPO_BETA_BASELINE})", dpo_policy),
    (f"DPO weak (beta={DPO_BETA_WEAK})", dpo_policy_weak),
]

ablation_results = []
for name, model in ablation_configs:
    mean_reward, _ = eval_mean_reward(model, EVAL_PROMPTS)
    diversity, degenerate_rate = eval_diversity(model, DIVERSITY_PROMPTS, DIVERSITY_K)
    ablation_results.append((name, mean_reward, diversity, degenerate_rate))
    print(f"{name:32s} reward={mean_reward:.3f}  unique-completion ratio={diversity:.3f}  "
          f"degenerate (single-repeated-digit) rate={degenerate_rate:.3f}")

In [ ]:
names = [r[0] for r in ablation_results]
rewards_plot = [r[1] for r in ablation_results]
diversity_plot = [r[2] for r in ablation_results]
degenerate_plot = [r[3] for r in ablation_results]
bar_colors = ["gray", "tab:blue", "tab:red", "tab:green", "tab:orange"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, values, title in zip(
    axes, [rewards_plot, diversity_plot, degenerate_plot],
    ["mean reward", "unique-completion ratio\n(diversity)", "degenerate rate\n(single repeated digit)"],
):
    ax.bar(range(len(names)), values, color=bar_colors)
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_title(title)
plt.tight_layout()
plt.show()

In [ ]:
# The ablation's core claim: for BOTH algorithms, weakening the regularization (PPO's kl_coef -> 0, DPO's
# beta -> a small value) causes a measurable collapse in output diversity relative to that algorithm's own
# well-regularized baseline. At SMOKE_TEST scale there are too few training steps for this effect to show
# up reliably (the whole point of SMOKE_TEST is a fast structural check, not a faithful reproduction of the
# full training dynamics), so the hard comparison is only asserted at full scale; the measured numbers
# above are printed either way.
ref_name, ref_reward, ref_diversity, ref_degenerate = ablation_results[0]
_, ppo_base_reward, ppo_base_diversity, ppo_base_degenerate = ablation_results[1]
_, ppo_weak_reward, ppo_weak_diversity, ppo_weak_degenerate = ablation_results[2]
_, dpo_base_reward, dpo_base_diversity, dpo_base_degenerate = ablation_results[3]
_, dpo_weak_reward, dpo_weak_diversity, dpo_weak_degenerate = ablation_results[4]

if not SMOKE_TEST:
    assert ppo_weak_diversity < ppo_base_diversity, (
        "expected PPO with no KL penalty to have LOWER output diversity than the regularized PPO baseline"
    )
    assert ppo_weak_degenerate > ppo_base_degenerate, (
        "expected PPO with no KL penalty to degenerate to repeated-digit completions more often"
    )
    assert dpo_weak_diversity < dpo_base_diversity, (
        "expected DPO with a small beta to have LOWER output diversity than the regularized DPO baseline"
    )
    assert dpo_weak_degenerate > dpo_base_degenerate, (
        "expected DPO with a small beta to degenerate to repeated-digit completions more often"
    )
    print("ablation assertions passed: weakening regularization measurably reduced diversity and "
          "increased the degenerate (repeated-digit) rate for BOTH PPO and DPO")
else:
    print("SMOKE_TEST: ablation ran end-to-end (numbers printed above); the full-scale run is what "
          "demonstrates the effect reliably enough to assert on.")

In [ ]:
# Final check, covering every training run in this notebook (PPO baseline, DPO baseline, PPO weak, DPO
# weak): pi_ref's parameters are still bit-for-bit identical to the snapshot taken right after freezing it.
for k, v in pi_ref.state_dict().items():
    assert torch.equal(v, pi_ref_snapshot[k]), f"pi_ref parameter '{k}' changed somewhere in this notebook!"
print("Final check: pi_ref is still exactly the frozen checkpoint from Part D, after every training run "
      "in this notebook (PPO baseline, DPO baseline, PPO weak ablation, DPO weak ablation).")

## Part J: summary

This notebook built an RLHF-style post-training pipeline from scratch on a tiny, fully verifiable task:

- A **rule-based reward function** (RLVR-style: no learned reward model) that is nonetheless *gameable* --
  a constant-digit run scores the same maximum reward as a genuinely ascending run, by design, so that
  reward hacking has somewhere real to happen.
- **PPO**: on-policy rollouts from the current policy, a clipped surrogate objective (genuinely
  `min(ratio * A, clip(ratio, 1-eps, 1+eps) * A)`, verified against hand-computed values for both positive
  and negative advantages), plus a KL penalty against a frozen reference policy, estimated via the per-token
  log-probability ratio the way practical large-vocabulary RLHF implementations do.
- **DPO**: the same frozen reference policy, but no rollouts and no reward-function calls during training --
  just a classification-style loss over static (chosen, rejected) preference pairs, derived from and
  matching the closed-form DPO objective.
- **A head-to-head comparison** from the identical starting checkpoint and a matched gradient-step budget,
  showing DPO making more consistent progress than a PPO run regularized strongly enough to avoid collapse
  at this toy scale and budget -- itself a small, real illustration of DPO's practical appeal over on-policy
  RLHF.
- **A genuine, measured reward-hacking / mode-collapse effect**: weakening PPO's KL coefficient toward zero
  and DPO's beta toward zero both reliably increase the model's raw reward *and* collapse its output
  diversity toward the exploitable repeated-digit loophole in the reward function -- exactly the failure
  mode that KL penalties (PPO) and beta (DPO) exist to prevent, and exactly why RLVR reward functions still
  need to be designed carefully even though they sidestep a learned, hackable reward *model*.